# Synthetic Heston Data

This notebook generates training data for the GP emulator. It samples plausible Heston parameter sets with Latin Hypercube sampling, prices option surfaces with the Carr-Madan FFT pricer, converts prices to Black-Scholes implied volatility, and saves the resulting table for model training.

In [ ]:
import csv
from pathlib import Path

import numpy as np
from scipy.interpolate import interp1d
from scipy.optimize import brentq
from scipy.stats import norm, qmc


In [ ]:
# Market inputs. These can be replaced with the SPY snapshot values later.
S0 = 100.0
RISK_FREE_RATE = 0.03
DIVIDEND_YIELD = 0.00

# 1,500 parameter sets stays within the requested 1,000-3,000 range.
N_PARAMETER_SETS = 1_500
RANDOM_SEED = 42

# The grid defines the strike and maturity points observed for every Heston draw.
MONEYNESS_GRID = np.linspace(0.70, 1.30, 13)
MATURITY_GRID = np.array([2, 4, 7, 10, 13, 21, 30, 60, 90, 180, 365], dtype=float) / 365.0

# Plausible variance-process ranges for synthetic training data.
PARAMETER_RANGES = {
    "v0": (0.01, 0.25),
    "kappa": (0.10, 5.00),
    "theta": (0.01, 0.25),
    "sigma_v": (0.05, 1.00),
    "rho": (-0.95, 0.00),
}

OUTPUT_PATH = Path("data/synthetic_heston_iv.csv")


## 1. Sample Heston Parameters

Latin Hypercube sampling spreads points across each parameter range more evenly than independent uniform draws, which is useful when the synthetic data will train a nonparametric emulator.

In [ ]:
def sample_heston_parameters(n_samples, parameter_ranges, seed=42):
    """Sample Heston parameters with Latin Hypercube sampling."""
    names = list(parameter_ranges)
    bounds = np.array([parameter_ranges[name] for name in names], dtype=float)

    sampler = qmc.LatinHypercube(d=len(names), seed=seed)
    unit_samples = sampler.random(n=n_samples)
    scaled_samples = qmc.scale(unit_samples, bounds[:, 0], bounds[:, 1])

    samples = []
    for parameter_set_id, values in enumerate(scaled_samples):
        row = {name: value for name, value in zip(names, values)}
        row["parameter_set_id"] = parameter_set_id

        # This diagnostic is useful later when studying stable vs unstable regions.
        row["feller_margin"] = 2.0 * row["kappa"] * row["theta"] - row["sigma_v"] ** 2
        samples.append(row)

    return samples


parameter_samples = sample_heston_parameters(N_PARAMETER_SETS, PARAMETER_RANGES, RANDOM_SEED)
parameter_samples[:5]


[{'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'parameter_set_id': 0,
  'feller_margin': 0.7203816367892698},
 {'v0': 0.05848390042373812,
  'kappa': 2.6879802769734984,
  'theta': 0.08507422971115568,
  'sigma_v': 0.1290855280326388,
  'rho': -0.5139185777606672,
  'parameter_set_id': 1,
  'feller_margin': 0.4406926295371334},
 {'v0': 0.04002067231612278,
  'kappa': 4.190105901036428,
  'theta': 0.029576981580787094,
  'sigma_v': 0.9152455843115952,
  'rho': -0.47084749565925726,
  'parameter_set_id': 2,
  'feller_margin': -0.5898131094888699},
 {'v0': 0.025643641804514436,
  'kappa': 3.1427216896957484,
  'theta': 0.06534978923902332,
  'sigma_v': 0.273042500257738,
  'rho': -0.3626667207861106,
  'parameter_set_id': 3,
  'feller_margin': 0.33620019317005195},
 {'v0': 0.030198705961586338,
  'kappa': 1.516575215170776,
  'theta': 0.11576468831609682,
  'sigma_v': 0.4712343566

## 2. Pricing and Implied Volatility Helpers

The Carr-Madan FFT pricer evaluates a full log-strike grid for one maturity at a time. The code below interpolates that grid to the strikes needed for the training surface, then inverts each price to Black-Scholes implied volatility.

In [ ]:
def black_scholes_call(S0, K, T, r, q, sigma):
    """Closed-form Black-Scholes call price."""
    if T <= 0:
        return max(S0 - K, 0.0)

    vol_sqrt_T = sigma * np.sqrt(T)
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma**2) * T) / vol_sqrt_T
    d2 = d1 - vol_sqrt_T
    return S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_volatility_call(price, S0, K, T, r, q, low=1e-4, high=5.0):
    """Invert a call price to Black-Scholes implied volatility."""
    intrinsic = max(S0 * np.exp(-q * T) - K * np.exp(-r * T), 0.0)
    upper_bound = S0 * np.exp(-q * T)

    # Invalid prices are kept as NaN and filtered before saving.
    if not np.isfinite(price) or price <= intrinsic or price >= upper_bound:
        return np.nan

    objective = lambda sigma: black_scholes_call(S0, K, T, r, q, sigma) - price

    try:
        return brentq(objective, low, high, maxiter=100)
    except ValueError:
        return np.nan


def heston_log_price_cf(u, S0, T, r, q, kappa, theta, sigma_v, rho, v0):
    """Characteristic function of log(S_T) under the Heston model."""
    u = np.asarray(u, dtype=complex)
    x0 = np.log(S0)
    iu = 1j * u

    a = kappa * theta
    b = kappa - rho * sigma_v * iu
    d = np.sqrt(b**2 + sigma_v**2 * (u**2 + iu))
    g = (b - d) / (b + d)
    exp_dt = np.exp(-d * T)

    # Little Heston Trap form improves stability of the complex logarithm.
    C = iu * (x0 + (r - q) * T) + (a / sigma_v**2) * (
        (b - d) * T - 2.0 * np.log((1.0 - g * exp_dt) / (1.0 - g))
    )
    D = ((b - d) / sigma_v**2) * ((1.0 - exp_dt) / (1.0 - g * exp_dt))
    return np.exp(C + D * v0)


def carr_madan_fft_call_grid(S0, T, r, q, params, alpha=1.5, n_fft=2**11, eta=0.25):
    """Price calls on a log-strike grid using the Carr-Madan FFT method."""
    j = np.arange(n_fft)
    v = eta * j
    lam = 2.0 * np.pi / (n_fft * eta)
    b = 0.5 * n_fft * lam
    k = -b + lam * j

    weights = np.ones(n_fft)
    weights[0] = 1.0 / 3.0
    weights[1::2] = 4.0 / 3.0
    weights[2::2] = 2.0 / 3.0

    u = v - 1j * (alpha + 1.0)
    phi = heston_log_price_cf(u, S0, T, r, q, **params)
    denominator = alpha**2 + alpha - v**2 + 1j * (2.0 * alpha + 1.0) * v
    psi = np.exp(-r * T) * phi / denominator

    fft_input = np.exp(1j * b * v) * psi * eta * weights
    fft_values = np.fft.fft(fft_input).real
    strikes = np.exp(k)
    calls = np.exp(-alpha * k) * fft_values / np.pi

    valid = np.isfinite(calls) & (calls > 0.0)
    return strikes[valid], calls[valid]


def carr_madan_fft_call(strikes, S0, T, r, q, params):
    """Interpolate FFT call prices to the requested strikes."""
    grid_strikes, grid_calls = carr_madan_fft_call_grid(S0, T, r, q, params)
    interpolator = interp1d(grid_strikes, grid_calls, kind="cubic", bounds_error=False, fill_value="extrapolate")
    return np.asarray(interpolator(strikes), dtype=float)


## 3. Generate the Synthetic Surface

Each row of the output is one training example: five Heston parameters plus one strike and maturity, mapped to the corresponding call price and implied volatility.

In [ ]:
def build_synthetic_dataset(parameter_samples, moneyness_grid, maturity_grid, S0, r, q):
    """Run the FFT pricer over sampled Heston parameters and option grids."""
    strikes = S0 * moneyness_grid
    rows = []

    for sample in parameter_samples:
        params = {
            "v0": sample["v0"],
            "kappa": sample["kappa"],
            "theta": sample["theta"],
            "sigma_v": sample["sigma_v"],
            "rho": sample["rho"],
        }

        for maturity in maturity_grid:
            prices = carr_madan_fft_call(strikes, S0, maturity, r, q, params)

            for moneyness, strike, price in zip(moneyness_grid, strikes, prices):
                implied_vol = implied_volatility_call(price, S0, strike, maturity, r, q)
                rows.append({
                    "parameter_set_id": int(sample["parameter_set_id"]),
                    "v0": sample["v0"],
                    "kappa": sample["kappa"],
                    "theta": sample["theta"],
                    "sigma_v": sample["sigma_v"],
                    "rho": sample["rho"],
                    "feller_margin": sample["feller_margin"],
                    "strike": strike,
                    "moneyness": moneyness,
                    "maturity": maturity,
                    "call_price": price,
                    "implied_vol": implied_vol,
                })

    return [row for row in rows if np.isfinite(row["call_price"]) and np.isfinite(row["implied_vol"])]


synthetic_data = build_synthetic_dataset(
    parameter_samples=parameter_samples,
    moneyness_grid=MONEYNESS_GRID,
    maturity_grid=MATURITY_GRID,
    S0=S0,
    r=RISK_FREE_RATE,
    q=DIVIDEND_YIELD,
)

synthetic_data[:5]


[{'parameter_set_id': 0,
  'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'feller_margin': 0.7203816367892698,
  'strike': 95.0,
  'moneyness': 0.95,
  'maturity': 0.005479452054794521,
  'call_price': 5.019753114989762,
  'implied_vol': 0.278983911568148},
 {'parameter_set_id': 0,
  'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'feller_margin': 0.7203816367892698,
  'strike': 100.0,
  'moneyness': 1.0,
  'maturity': 0.005479452054794521,
  'call_price': 0.8204487579057981,
  'implied_vol': 0.2750613524822858},
 {'parameter_set_id': 0,
  'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'feller_margin': 0.7203816367892698,
  'strike': 105.0,
  'moneyness': 1.05,
  'matu

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

fieldnames = list(synthetic_data[0])
with OUTPUT_PATH.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(synthetic_data)

print(f"Saved {len(synthetic_data):,} rows to {OUTPUT_PATH}")
print(f"Parameter sets sampled: {len(parameter_samples):,}")
print(f"Valid parameter sets in output: {len({row['parameter_set_id'] for row in synthetic_data}):,}")

implied_vols = np.array([row["implied_vol"] for row in synthetic_data])
print(f"Implied vol range: {implied_vols.min():.4f} to {implied_vols.max():.4f}")


Saved 182,016 rows to data/synthetic_heston_iv.csv
Parameter sets sampled: 1,500
Valid parameter sets in output: 1,500
Implied vol range: 0.0538 to 3.7582
